# Trending Topic & Viral Signal Forecaster

A proof-of-concept **agentic system** that monitors live information channels, compares emerging signals against accumulated history, and forecasts which topics will gain traction in the next 24-48 hours.

This notebook is the *runner*. The agentic logic lives in the helper `.py` files:

- `agents.py` — the four agents (Collector, Retrieval, Critic, Forecast) + Controller
- `vector_store.py` — long-term memory / RAG layer
- `tools.py` — live data retrieval + deterministic signal scoring
- `llm_clients.py` — Claude (reasoning) and Gemini (embeddings)
- `guardrails.py` — safety checks and human-review triggers
- `config.py` — all tuning constants

## 1. Setup

Confirm the two API keys are present. `ANTHROPIC_API_KEY` drives the reasoning agents; `GEMINI_API_KEY` powers semantic retrieval (falls back to a local embedding if absent).

In [1]:
import config

print("Claude key set:", bool(config.ANTHROPIC_API_KEY))
print("Gemini key set:", bool(config.GEMINI_API_KEY))
print("Reasoning model:", config.CLAUDE_MODEL)
print("Embedding model:", config.GEMINI_EMBED_MODEL)

Claude key set: True
Gemini key set: True
Reasoning model: claude-opus-4-8
Embedding model: gemini-embedding-001


## 2. Initialise long-term memory (the RAG vector store)

**Agentic feature — long-term memory (checkpoint 3).** The vector store is the agent's persistent knowledge of past topic trajectories and their outcomes. We seed a few synthetic records so the very first cycle has historical analogues to retrieve.

In [2]:
from vector_store import VectorStore

store = VectorStore()
store.seed_demo_data()
print(f"Records in long-term memory: {len(store.records)}")

[vector_store] seeded 5 demo records.
Records in long-term memory: 5


## 3. Build the multi-agent system

**Agentic feature — multi-agent architecture (checkpoint 5).** The Controller wires together the four specialised agents and runs them in the *sequential-with-feedback* pattern:

`Collector -> Retrieval -> Critic (Tree-of-Thought) -> Forecast`

In [3]:
from agents import ForecastController

controller = ForecastController(store=store)
print("Agents ready:")
print(" - Collector (live data + topic extraction)")
print(" - Retrieval (semantic RAG)")
print(" - Critic    (Tree-of-Thought beam search)")
print(" - Forecast  (briefing + write-back)")

Agents ready:
 - Collector (live data + topic extraction)
 - Retrieval (semantic RAG)
 - Critic    (Tree-of-Thought beam search)
 - Forecast  (briefing + write-back)


## 4. Run one full forecast cycle

This executes the whole **ReAct-style loop** end to end (checkpoint 2): the Collector *acts* (fetches data) and *observes* (validates it); the Critic *reasons* over multiple interpretations via Tree-of-Thought (checkpoint 4); and a feedback hop re-queries memory when retrieved analogues look weak.

In [4]:
cycle = controller.run_cycle(verbose=True)
print("\nCycle OK:", cycle["ok"])

[1/4] Collector: fetching live data...
[reddit] failed to fetch r/technology: 403 Client Error: Blocked for url: https://www.reddit.com/r/technology/hot.json?limit=20
[reddit] failed to fetch r/worldnews: 403 Client Error: Blocked for url: https://www.reddit.com/r/worldnews/hot.json?limit=20
[reddit] failed to fetch r/news: 403 Client Error: Blocked for url: https://www.reddit.com/r/news/hot.json?limit=20
[reddit] failed to fetch r/science: 403 Client Error: Blocked for url: https://www.reddit.com/r/science/hot.json?limit=20
  80 items -> 8 candidate topics
[2/4] Retrieval: querying long-term memory...
[3/4] Critic: Tree-of-Thought scoring...
[4/4] Forecast: generating briefings...

Cycle OK: True


## 5. Review the ranked forecasts

Each forecast carries its **evidence chain** (output constraint, checkpoint 6): the live signal, the retrieved historical analogues, and the winning interpretation. Topics may be `RELEASED` or `HELD_FOR_REVIEW` by the guardrails.

In [5]:
if cycle["ok"]:
    for r in cycle["results"]:
        print("=" * 70)
        print(f"TOPIC      : {r['topic']}")
        print(f"CONFIDENCE : {r['confidence']}")
        print(f"STATUS     : {r['status']}")
        if r["review"]["requires_review"]:
            print(f"REVIEW     : {r['review']['reasons']}")
        print(f"BRIEFING   : {r['briefing']}")
else:
    print("Cycle did not produce results:", cycle["detail"])

TOPIC      : AI, apps and cybersecurity
CONFIDENCE : 0.94
STATUS     : RELEASED
BRIEFING   : **Forecast Briefing: AI, Apps & Cybersecurity**

Strong developer and early-adopter engagement (1,255 interactions) on Hacker News and RSS signals high likelihood of mainstream tech media pickup within 24-48 hours, with confidence at 0.94. The main driver is the combination of high velocity on a tastemaker platform and the evergreen, high-salience nature of this theme, which journalists actively monitor. Key uncertainty: historical analogues are split—concrete events (breaches, hardware launches) tend to go viral, while abstract discussion-based topics (e.g., AI regulation debates) often fade, so the specific framing of this story will determine whether it cascades or stalls.
TOPIC      : Tech hardware: chips, Apple, Xbox, Framework, RAM
CONFIDENCE : 0.84
STATUS     : HELD_FOR_REVIEW
REVIEW     : ["content_sensitivity: ['war']"]
BRIEFING   : **Forecast Briefing: Tech Hardware (Chips, Apple, Xbo

## 6. Inspect the evidence chain for the top forecast

This is what makes the agent auditable rather than a black box — every score is traceable to live signals and retrieved analogues.

In [6]:
import json

if cycle["ok"] and cycle["results"]:
    top = cycle["results"][0]
    print(json.dumps(top["evidence_chain"], indent=2)[:2500])

{
  "live_signal": {
    "platforms": [
      "hackernews",
      "rss"
    ],
    "velocity": 1.0,
    "engagement": 1255
  },
  "retrieved_analogues": [
    {
      "summary": "New AI regulation bill debated in legislature",
      "metadata": {
        "platforms": [
          "reddit"
        ],
        "velocity": 0.7
      },
      "outcome": "faded",
      "timestamp": "2026-06-26T05:08:38.879030+00:00",
      "similarity": 0.629
    },
    {
      "summary": "AI regulation discussion spikes in tech communities",
      "metadata": {
        "platforms": [
          "reddit"
        ],
        "velocity": 0.65
      },
      "outcome": "faded",
      "timestamp": "2026-06-26T05:08:39.251761+00:00",
      "similarity": 0.609
    },
    {
      "summary": "Major data breach at large retailer",
      "metadata": {
        "platforms": [
          "reddit",
          "rss"
        ],
        "velocity": 0.8
      },
      "outcome": "went_viral",
      "timestamp": "2026-06-26T05:08:3

## 7. Memory grows across cycles

**Agentic feature — write-back / feedback loop (checkpoints 1 & 4).** The Forecast Agent wrote each observed topic back to long-term memory. Over many cycles this is what lets the agent distinguish a genuinely novel signal from a recurring pattern.

In [7]:
print(f"Records in long-term memory now: {len(store.records)}")
print("Most recent record summary:", store.records[-1]["summary"])

Records in long-term memory now: 13
Most recent record summary: China, Taiwan and AI rivalry
